---
title: "Module 2 Assignment: Spark SQL and DataFrames"
author:
  - name: Julie NGO TRAN
    affiliations:
      - id: bu
        name: Boston University
        city: Boston
        state: MA
number-sections: true
date: '2026-06-02'
format:
  html:
    theme: cerulean
    toc: true
    toc-depth: 2
date-modified: today
date-format: long
execute:
  echo: true
  eval: true
  freeze: auto
---

## System Information {.unnumbered}

In [ ]:
#| eval: true
#| echo: true
import platform
import psutil
import subprocess

print("Operating System:", platform.system(), platform.release())
print("Python Version:", platform.python_version())
print("Machine:", platform.machine())

# EC2 Linux doesn't expose processor via platform.processor()
# Use lscpu instead
try:
    processor = subprocess.check_output("lscpu | grep 'Model name'", shell=True).decode().strip()
    processor = processor.split(":")[1].strip()
except:
    processor = "Unable to retrieve"
print("Processor:", processor)

mem = psutil.virtual_memory()
print("Total Memory (MB):", round(mem.total / (1024**2), 2))
print("Available Memory (MB):", round(mem.available / (1024**2), 2))

## Step 1: Load the Dataset into Spark

In [ ]:
#| echo: true
#| eval: true

from pyspark.sql import SparkSession

# Start a Spark session
spark = SparkSession.builder.config("spark.driver.host", "localhost").appName("JobPostingsAnalysis").getOrCreate()
spark.catalog.clearCache()

# Load the CSV file into a Spark DataFrame
df = spark.read.option("header", "true").option("inferSchema", "true").option(
    "multiLine", "true").option("escape", "\"").csv("data/lightcast_job_postings.csv")

# Register the DataFrame as a temporary SQL view
df.createOrReplaceTempView("job_postings")

# comment the lines below when rendering the submission
# df.printSchema()
# df.show(5)

## Step 2: Create Relational Tables in Spark SQL

### Industries Table

In [ ]:
#| echo: true
#| eval: true

from pyspark.sql.functions import col, monotonically_increasing_id

industries_df = df.select(
    col("NAICS_2022_6"),
    col("NAICS_2022_6_NAME"),
    col("SOC_5"),
    col("SOC_5_NAME"),
    col("LOT_SPECIALIZED_OCCUPATION_NAME"),
    col("LOT_OCCUPATION_GROUP")
).distinct().withColumn("industry_id", monotonically_increasing_id())

industries_df = industries_df.select(
    "industry_id",
    "NAICS_2022_6",
    "NAICS_2022_6_NAME",
    "SOC_5",
    "SOC_5_NAME",
    "LOT_SPECIALIZED_OCCUPATION_NAME",
    "LOT_OCCUPATION_GROUP"
)

# Save as CSV
industries_df.write.mode("overwrite").csv("output/industries.csv", header=True)

# Register as SQL view
industries_df.createOrReplaceTempView("industries")
industries_df.show(5, truncate=False)

# Pandas conversion for display
industries_df.toPandas()

### Companies Table

In [ ]:
#| echo: true
#| eval: true

companies_df = df.select(
    col("COMPANY"),
    col("COMPANY_NAME"),
    col("COMPANY_RAW"),
    col("COMPANY_IS_STAFFING")
).distinct().withColumn("company_id", monotonically_increasing_id())

companies_df = companies_df.select(
    "company_id",
    "COMPANY",
    "COMPANY_NAME",
    "COMPANY_RAW",
    "COMPANY_IS_STAFFING"
)

# Save as CSV
companies_df.write.mode("overwrite").csv("output/companies.csv", header=True)

# Register as SQL view
companies_df.createOrReplaceTempView("companies")
companies_df.show(5, truncate=False)

# Pandas conversion for display
companies_df.toPandas()

### Locations Table

In [ ]:
#| echo: true
#| eval: true

locations_df = df.select(
    col("LOCATION"),
    col("CITY_NAME"),
    col("STATE_NAME"),
    col("COUNTY_NAME"),
    col("MSA"),
    col("MSA_NAME")
).distinct().withColumn("location_id", monotonically_increasing_id())

locations_df = locations_df.select(
    "location_id",
    "LOCATION",
    "CITY_NAME",
    "STATE_NAME",
    "COUNTY_NAME",
    "MSA",
    "MSA_NAME"
)

# Save as CSV
locations_df.write.mode("overwrite").csv("output/locations.csv", header=True)

# Register as SQL view
locations_df.createOrReplaceTempView("locations")
locations_df.show(5, truncate=False)

# Pandas conversion for display
locations_df.toPandas()

### Job Postings Table

In [ ]:
#| echo: true
#| eval: true

df_with_id = df.withColumnRenamed("COMPANY", "jp_COMPANY") \
               .withColumnRenamed("COMPANY_NAME", "jp_COMPANY_NAME") \
               .withColumnRenamed("NAICS_2022_6", "jp_NAICS_2022_6") \
               .withColumnRenamed("SOC_5", "jp_SOC_5") \
               .withColumnRenamed("CITY_NAME", "jp_CITY_NAME") \
               .withColumnRenamed("STATE_NAME", "jp_STATE_NAME")

job_postings_df = df_with_id \
.join(companies_df, df_with_id["jp_COMPANY"] == companies_df["COMPANY"], "left") \
.join(industries_df,
    (df_with_id["jp_NAICS_2022_6"] == industries_df["NAICS_2022_6"]) &
    (df_with_id["jp_SOC_5"] == industries_df["SOC_5"]),
    "left") \
.join(locations_df,
    (df_with_id["jp_CITY_NAME"] == locations_df["CITY_NAME"]) &
    (df_with_id["jp_STATE_NAME"] == locations_df["STATE_NAME"]),
    "left") \
.select(
    df_with_id["ID"],
    df_with_id["TITLE_CLEAN"],
    companies_df["company_id"],
    industries_df["industry_id"],
    df_with_id["EMPLOYMENT_TYPE_NAME"],
    df_with_id["REMOTE_TYPE_NAME"],
    df_with_id["BODY"],
    df_with_id["MIN_YEARS_EXPERIENCE"],
    df_with_id["MAX_YEARS_EXPERIENCE"],
    df_with_id["SALARY"],
    df_with_id["SALARY_FROM"],
    df_with_id["SALARY_TO"],
    locations_df["location_id"],
    df_with_id["POSTED"],
    df_with_id["EXPIRED"],
    df_with_id["DURATION"]
)

# Register as SQL view
job_postings_df.createOrReplaceTempView("job_postings")
job_postings_df.show(5, truncate=False)

## Step 3: Run SQL Queries in Spark

### Example Query: Top 5 Most Posted Job Titles

In [ ]:
#| echo: true
#| eval: true

top5 = spark.sql("""
    SELECT TITLE_CLEAN, COUNT(*) AS job_count
    FROM job_postings
    GROUP BY TITLE_CLEAN
    ORDER BY job_count DESC
    LIMIT 5
""")
top5.show(truncate=False)
top5.toPandas()

### Query 1: Average Salary by Tech Industry

> **Note on Query 1 Approach:** The Lightcast dataset stores salary data as
> pre-aggregated industry-level values, meaning all occupations within the same
> NAICS code share identical salary figures. Therefore, instead of grouping by
> specialized occupation as instructed, we group by NAICS industry code across
> five major tech sectors (518210, 541511, 541512, 541519, 334111) to produce
> meaningful salary variation. This reflects a data limitation of Lightcast's
> pre-summarized compensation model rather than an analytical choice.

In [ ]:
#| echo: true
#| eval: true

query1 = spark.sql("""
    SELECT 
        i.NAICS_2022_6_NAME AS industry_name,
        ROUND(AVG((j.SALARY_FROM + j.SALARY_TO) / 2), 2) AS avg_salary,
        COUNT(DISTINCT j.ID) AS job_count
    FROM job_postings j
    JOIN industries i ON j.industry_id = i.industry_id
    WHERE i.NAICS_2022_6 IN (518210, 541511, 541512, 541519, 334111)
    AND j.SALARY_FROM IS NOT NULL
    AND j.SALARY_TO IS NOT NULL
    AND j.SALARY_FROM > 0
    AND j.SALARY_TO > 0
    GROUP BY i.NAICS_2022_6_NAME
    ORDER BY avg_salary DESC
""")

query1.show(truncate=False)
query1_pd = query1.toPandas()
query1_pd

In [ ]:
#| echo: true
#| eval: true

import plotly.express as px
import textwrap

query1_pd["industry_name"] = query1_pd["industry_name"].apply(
    lambda x: "<br>".join(textwrap.wrap(x, width=40))
)

fig = px.bar(query1_pd,
    x="avg_salary",
    y="industry_name",
    color="industry_name",
    orientation="h",
    title="Average Salary by Tech Industry (NAICS 2022)",
    labels={"avg_salary": "Average Salary (USD)", "industry_name": "Industry"},
    color_discrete_sequence=px.colors.qualitative.Set2)

fig.update_layout(
    showlegend=False,
    margin=dict(l=350, r=100, t=80, b=50),
    yaxis=dict(tickfont=dict(size=11)),
    xaxis=dict(range=[0, 200000], tickformat="$,.0f"),
    height=400
)

fig.update_traces(
    texttemplate="%{x:$,.0f}",
    textposition="outside"
)

fig.show()

**Insight:** Electronic Computer Manufacturing ($167,149) offers the highest average salary among tech industries, significantly higher than Custom Computer Programming Services ($116,847). This suggests hardware manufacturing roles command premium compensation compared to software services, likely due to specialized skills and lower job volume.

### Query 2: Top Companies with Most Remote Jobs in California, Washington and New York

In [ ]:
#| echo: true
#| eval: true

query2 = spark.sql("""
    SELECT 
        c.COMPANY_NAME AS company_name,
        l.STATE_NAME AS state,
        COUNT(DISTINCT j.ID) AS remote_jobs
    FROM job_postings j
    JOIN companies c ON j.company_id = c.company_id
    JOIN locations l ON j.location_id = l.location_id
    WHERE j.REMOTE_TYPE_NAME = 'Remote'
    AND l.STATE_NAME IN ('California', 'Washington', 'New York')
    AND c.COMPANY_NAME != 'Unclassified'
    GROUP BY c.COMPANY_NAME, l.STATE_NAME
    ORDER BY remote_jobs DESC
    LIMIT 15
""")

query2.show(truncate=False)
query2_pd = query2.toPandas()
query2_pd

In [ ]:
#| echo: true
#| eval: true

query2_pd_sorted = query2_pd.sort_values(["company_name", "state"])

fig = px.bar(query2_pd_sorted,
    x="remote_jobs",
    y="company_name",
    color="state",
    barmode="group",
    orientation="h",
    title="Top Companies with Remote Jobs in California, Washington and New York<br>(June - September 2024)",
    labels={"company_name": "Company", "remote_jobs": "Remote Job Postings", "state": "State"},
    color_discrete_sequence=px.colors.qualitative.Set2)

fig.update_layout(
    margin=dict(l=300, r=100, t=100, b=50),
    height=550,
    yaxis=dict(tickfont=dict(size=11))
)

fig.show()

**Insight:** Kaiser Permanente leads remote job postings in California with 33 postings, followed by University of California with 23. Washington state shows strong presence from BCforward and Providence, while New York is represented by The Judge Group. Healthcare and staffing companies dominate remote hiring across all three states.

### Query 3: Monthly Job Posting Trends in California, Washington and New York

In [ ]:
#| echo: true
#| eval: true

query3 = spark.sql("""
    SELECT 
        YEAR(to_date(j.POSTED, 'M/d/yyyy')) AS year,
        MONTH(to_date(j.POSTED, 'M/d/yyyy')) AS month,
        l.STATE_NAME AS state,
        COUNT(DISTINCT j.ID) AS job_count
    FROM job_postings j
    JOIN locations l ON j.location_id = l.location_id
    WHERE l.STATE_NAME IN ('California', 'Washington', 'New York')
    AND to_date(j.POSTED, 'M/d/yyyy') IS NOT NULL
    GROUP BY year, month, l.STATE_NAME
    ORDER BY year ASC, month ASC
""")

query3.show(truncate=False)
query3_pd = query3.toPandas()
query3_pd

In [ ]:
#| echo: true
#| eval: true

query3_pd["month_year"] = query3_pd["month"].astype(str) + "/" + query3_pd["year"].astype(str)

fig = px.line(query3_pd,
    x="month_year",
    y="job_count",
    color="state",
    title="Monthly Job Posting Trends in California, Washington and New York<br>(June - September 2024)",
    labels={"month_year": "Month/Year", "job_count": "Job Postings", "state": "State"},
    markers=True,
    color_discrete_sequence=px.colors.qualitative.Set2)

fig.update_layout(
    margin=dict(l=50, r=50, t=100, b=50),
    height=450
)

fig.show()

**Insight:** California consistently dominates job postings across all months, peaking in June 2024. Both California and Washington show a dip in August before recovering slightly in September. New York shows steady but lower posting volume compared to California throughout the period.

### Query 4: Salary Comparisons Across Major US Cities

In [ ]:
#| echo: true
#| eval: true

query4 = spark.sql("""
    SELECT 
        CASE l.MSA
            WHEN 14460 THEN 'Boston'
            WHEN 47900 THEN 'Washington DC'
            WHEN 35620 THEN 'New York'
            WHEN 41860 THEN 'San Francisco'
            WHEN 42660 THEN 'Seattle'
            WHEN 31080 THEN 'Los Angeles'
            WHEN 19100 THEN 'Dallas'
            WHEN 26420 THEN 'Houston'
            WHEN 12420 THEN 'Austin'
            WHEN 34980 THEN 'Nashville'
            WHEN 28140 THEN 'Kansas City'
            WHEN 19740 THEN 'Denver'
        END AS city,
        ROUND(AVG(j.SALARY), 2) AS avg_salary,
        COUNT(DISTINCT j.ID) AS job_count
    FROM job_postings j
    JOIN locations l ON j.location_id = l.location_id
    WHERE j.SALARY IS NOT NULL
    AND j.SALARY > 0
    AND l.MSA IN (14460, 47900, 35620, 41860, 42660, 31080, 19100, 26420, 12420, 34980, 28140, 19740)
    GROUP BY l.MSA
    ORDER BY avg_salary DESC
""")

query4.show(truncate=False)
query4_pd = query4.toPandas()
query4_pd

In [ ]:
#| echo: true
#| eval: true

fig = px.bar(query4_pd,
    x="city",
    y="avg_salary",
    color="city",
    title="Average Salary Across Major US Cities",
    labels={"city": "City", "avg_salary": "Average Salary (USD)"},
    color_discrete_sequence=px.colors.qualitative.Set2)

fig.update_layout(
    showlegend=False,
    xaxis_tickangle=-45,
    yaxis=dict(tickformat="$,.0f"),
    margin=dict(l=50, r=50, t=80, b=150),
    height=500
)

fig.update_traces(
    texttemplate="%{y:$,.0f}",
    textposition="outside"
)

fig.show()

**Insight:** San Francisco leads with the highest average salary among major US metro areas, followed by Seattle and Boston. Southern cities like Nashville, Houston and Dallas show comparatively lower average salaries, reflecting regional cost-of-living differences and industry concentration patterns.